In [1]:
#### ------------------------------------------------------------------------------------------
#### author: Ranjan Barman, date: May 30, 2025 (modified for immune only)
#### Compute 12 NPIFs based on HoverNet prediction (immune nuclei only)
#### Uses MPP = 0.248 for unit conversion and IQR filtering
#### Computes across **all tiles** with immune nuclei
#### ------------------------------------------------------------------------------------------

import os
import pandas as pd
import numpy as np

# Set working directory
_wpath_ = "/data/Lab_ruppin/Ranjan/HnE/"
os.makedirs(_wpath_, exist_ok=True)
os.chdir(_wpath_)
print("Working directory:", _wpath_)

# Define dataset name and output file path
dataset_name = "TCGA_BRCA_FFPE"
input_folder = f"{dataset_name}/outputs/HoverNet/"
output_file_path = f"{dataset_name}/outputs/HoverNet/HoverNet_NPIFs_TCGA_BRCA_1106_AllTiles_ImmuneOnly.csv"

# NPIF feature columns
columns_to_compute = ["Area", "Major Axis", "Minor Axis", "Perimeter", "Eccentricity", "Circularity"]

# Microns-per-pixel
MPP = 0.248

# IQR-based outlier removal
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

# Store results
results = []

# Get all TCGA folders
tcga_folders = [f for f in os.listdir(input_folder) if os.path.isdir(os.path.join(input_folder, f)) and f.startswith("TCGA")]

# Process each slide
for slide_name in tcga_folders:
    file_path = os.path.join(input_folder, slide_name, "features3", f"{slide_name}.csv")
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}, skipping...")
        continue

    df = pd.read_csv(file_path)

    # Filter for immune nuclei only
    df = df[df["Nucleus Type"] == "Immune"]
    if df.empty:
        print(f"  - No immune nuclei found for {slide_name}, skipping...\n")
        continue

    # Unit conversion
    df["Area"] = df["Area"] * (MPP ** 2)
    df["Major Axis"] = df["Major Axis"] * MPP
    df["Minor Axis"] = df["Minor Axis"] * MPP
    df["Perimeter"] = df["Perimeter"] * MPP

    # Remove outliers
    df = remove_outliers(df, "Major Axis")
    df = remove_outliers(df, "Minor Axis")

    if df.empty:
        print(f"  - All immune nuclei removed as outliers for {slide_name}, skipping...\n")
        continue

    # Count tiles with at least one immune nucleus
    total_tiles_with_nuclei = df["Tile"].nunique()

    # Compute mean and std
    mean_values = df[columns_to_compute].mean()
    std_values = df[columns_to_compute].std()

    # Append results
    results.append([slide_name, total_tiles_with_nuclei] + mean_values.tolist() + std_values.tolist())

# Final result dataframe
result_df = pd.DataFrame(
    results,
    columns=["Slide_Name", "Total_Tiles"] + 
            [f"Mean {col}" for col in columns_to_compute] + 
            [f"Std {col}" for col in columns_to_compute]
)

# Clean up NaNs and infs
result_df.replace([np.inf, -np.inf], np.nan, inplace=True)
result_df.fillna(result_df.mean(numeric_only=True), inplace=True)
result_df.fillna(0, inplace=True)

# Save results
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
result_df.to_csv(output_file_path, index=False)

print(f"\nImmune-only NPIFs from all tiles saved to: {output_file_path}")


Working directory: /data/Lab_ruppin/Ranjan/HnE/

Immune-only NPIFs from all tiles saved to: TCGA_BRCA_FFPE/outputs/HoverNet/HoverNet_NPIFs_TCGA_BRCA_1106_AllTiles_ImmuneOnly.csv


In [2]:
result_df

,Slide_Name,Total_Tiles,Mean Area,Mean Major Axis,Mean Minor Axis,Mean Perimeter,Mean Eccentricity,Mean Circularity,Std Area,Std Major Axis,Std Minor Axis,Std Perimeter,Std Eccentricity,Std Circularity
0,TCGA-D8-A13Z-01Z-00-DX1_3624_tiles,2925,5.010444,3.046825,2.263014,8.759328,0.619727,0.810975,1.564680,0.604015,0.337090,1.529502,0.154737,0.071151
1,TCGA-AR-A0TR-01Z-00-DX1_3099_tiles,2698,4.793022,2.988032,2.200277,8.525638,0.628306,0.811276,1.882755,0.678778,0.385417,1.761129,0.150353,0.069306
2,TCGA-D8-A1JN-01Z-00-DX1_4957_tiles,4833,5.218525,3.036400,2.346849,8.810168,0.588048,0.830075,1.724749,0.569221,0.362416,1.481756,0.153650,0.057215
3,TCGA-A2-A0ES-01Z-00-DX1_4815_tiles,4688,5.632249,3.253630,2.367940,9.234350,0.640694,0.815879,1.944148,0.672999,0.383574,1.701399,0.147579,0.063694
4,TCGA-C8-A12U-01Z-00-DX1_4200_tiles,3768,4.822230,2.983113,2.216350,8.533871,0.620180,0.813978,1.861502,0.663230,0.388975,1.715919,0.152888,0.068128
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1101,TCGA-BH-A18S-01Z-00-DX1_1472_tiles,828,6.073237,3.367985,2.459421,9.617495,0.638118,0.807151,2.236841,0.719580,0.445410,1.872601,0.149185,0.067798
1102,TCGA-OK-A5Q2-01Z-00-DX2_4517_tiles,4309,5.607109,3.194973,2.382451,9.188137,0.615063,0.813347,2.331277,0.758456,0.450931,2.002011,0.154783,0.068354
1103,TCGA-EW-A1IX-01Z-00-DX1_2878_tiles,1719,5.099911,3.099397,2.250224,8.798581,0.642062,0.808342,2.044626,0.713416,0.415905,1.845601,0.146089,0.069935
1104,TCGA-B6-A0IO-01Z-00-DX1_2044_tiles,1868,5.881940,3.307956,2.414926,9.443570,0.638350,0.808563,2.297783,0.731980,0.444341,1.913320,0.147946,0.067426
